# Task 2 — Incremental CPG parser

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | `screenshots/stage2_manifest.json` |
| Command | `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode sample` |
| Parser scope | Five-file bounded sample run. |
| Stable identity | Repository, commit, file identity, node location/type, and edge endpoints. |
```

## Approach and reasoning

Command: `docker compose run --rm ... parser python -m parser_service.main --repo data/datasets --mode sample`. The parser emits AST, CFG, DFG, and CALL records with stable IDs so downstream consumers can merge nodes and relationships deterministically. The sample is intentionally bounded for Stage 2 evidence, while the ID strategy is designed to remain stable across repeated captures.

## What this chapter proves

| Requirement | Evidence in this chapter |
|---|---|
| Incremental CPG extraction | Executed output reports node and edge counts from the captured run. |
| Error handling | The run preserves the intentional syntax-error event instead of dropping it. |
| Deterministic IDs | The text explains the ID ingredients used by the parser and graph sinks. |


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
manifest = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())
m = manifest['metrics']['kafka']
print('AST nodes:', m['ast_nodes'])
print('CFG edges:', m['cfg_edges'])
print('DFG edges:', m['dfg_edges'])
print('CALL edges:', m['call_edges'])
print('total edges:', m['total_edges'])
print('metadata events:', m['metadata_events'], 'error events:', m['error_events'])


AST nodes: 21415
CFG edges: 3537
DFG edges: 2248
CALL edges: 2183
total edges: 7968
metadata events: 5 error events: 1


## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Five sample files produced AST, CFG, DFG, and CALL structures plus one intentional syntax-error event. |
| Failed | A previous capture sampled fewer metadata/error records than the assignment requires. |
| Resolution | Topic-specific capture limits now retain all five metadata events and the single controlled error. The parser remains intentionally bounded to five files for Stage 2; repository-wide scaling is outside this sample-run scope. |
```
